In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
from scipy.signal import medfilt


In [ ]:
#first, get list of participants that are in the study, from xls file 

path_excel = '../../2023_study2_data/ParticipantLog.xlsx'

df = pd.read_excel(path_excel, sheet_name='ParticipantLog')

participant_list = []
for ind,row in df.iterrows():
    if row['USE?'] in ['Y','M']:
        participant_list.append(row['participant_id'])
    #print(row['participant_id'], row['USE?'])

print(len(participant_list))
#participant_list

In [ ]:
#get session list, by joining every 2 participants in a single string
session_list = []
for i in range(0,len(participant_list),2):
    session_list.append(str(participant_list[i])+str(participant_list[i+1]))
print(len(session_list))
#session_list

# IDC CURVE ATTEMPTS

In [ ]:
#imports
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
import scipy.stats as stats
import itertools
from scipy.stats import pearsonr
from scipy.integrate import trapz
#import tikzplotlib#

#for plots
import matplotlib as mpl

sns.set(style="whitegrid", font_scale=1.2)#, font='Arial')

#sns.set_palette("husl")
#plt.rcParams.update({'font.size': '50'})
#plt.rcParams['font.size'] = '50'
#mpl.rcParams['font.family'] = 'Arial'

from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
#from termcolor import colored


In [ ]:
#open df_interp
df_interp = pd.read_pickle('df_interpolated5dec.pkl')


#trying this with 0.5s of interpolation, which seems like enough for a system monitoring IDC


In [ ]:
df_interp

In [ ]:
#plot just sessions 41 an 38, as a figure with 4 subplots

sns.set(style="ticks", font_scale=2)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']

#create folder figures
if not os.path.exists('figures'):
    os.makedirs('figures')


sessions = df_interp['session'].unique()
sessions = [22, 39]
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    fig, (ax1,ax2) = plt.subplots(2,1,figsize = (15,9))  

    x = np.array(df_session['frame_seconds'].values[0])
    y0 = np.array(df_session['value'].values[0])
    ax1.plot(x,y0,label = labs[0], color = colors0[0],linewidth=3)
    y1 = np.array(df_session['value'].values[1])
    ax1.plot(x,y1,label = labs[1], color = colors1[0],linewidth=3)
    #now, plot the cumulative sum
    y_cumsum0 = np.array(df_session['cumsum'].values[0])
    ax2.plot(x,y_cumsum0,label = labs[0], color = colors0[0],alpha = 1.0, linewidth=3)
    y_cumsum1 = np.array(df_session['cumsum'].values[1])
    ax2.plot(x,y_cumsum1,label = labs[1], color = colors1[0],alpha = 1.0, linewidth=3)

    y_cumsum_sum = y_cumsum0 + y_cumsum1
    y_cumsum_sum_time = y_cumsum_sum / x #WONDERING if we should have time as a unit rather than seconds
    
    y_dif = np.abs(y0 - y1)
    y_all = y_cumsum_sum_time - y_dif
    
    #for section 22, make box around seconds 15-80

    if session == 22:
        ax1.axvline(x=15, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        ax1.axvline(x=80, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        ax2.axvline(x=15, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        ax2.axvline(x=80, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        #ax1.axvspan(15, 80, color='gray', alpha=0.1)
        #ax2.axvspan(15, 80, color='gray', alpha=0.1)
    if session == 39: #make box around 194-248
        ax1.axvline(x=194, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        ax1.axvline(x=248, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        ax2.axvline(x=194, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        ax2.axvline(x=248, color='g', linestyle='-',linewidth=3, alpha = 0.7)
        #ax1.axvspan(194, 248, color='gray', alpha=0.1)
        #ax2.axvspan(194, 248, color='gray', alpha=0.1)
        

    #ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Interpersonal Rating')
    #set scale from -7 to 7
    ax1.set_ylim([-7,8])
    #set yticks
    ax1.set_yticks([-7,-3,0,3,7])
    #add horizontal dashed line at 0
    ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    #vertical lines in both, every 100 seconds
    for i in range(0,x.max().astype(int),100):
        ax1.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
        ax2.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
    #set minimum x value to 0, but no upper bound
    ax1.set_xlim([0,None])

    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Cumulative IR')
    #set minimum x value to 0, but no upper bound
    ax2.set_xlim([0,None])

   # ax3

    ax1.legend()

    #tight layout for both axes
    plt.tight_layout()


    #tq_temp['condition'] == 'datadriven']
    #    plt.boxplot(df_postq_tempdd[col], labels = ['DataL'], positions = [ind*2+0.9],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors1[0], color=colors1[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
    plt.tight_layout()
    #sns.despine()
    print(session)
    plt.show()  
    #save with high quality in png
    #fig.savefig('figures/supp' + str(session) + '.png', dpi=500)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
#plot just sessions 41 an 38, as a figure with 4 subplots
#import tikzplotlib

sns.set(style="ticks", font_scale=2)
#import itertools
#plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']

#create folder figures
if not os.path.exists('figures'):
    os.makedirs('figures')


sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    fig, (ax1,ax2) = plt.subplots(2,1,figsize = (15,10))  

    x = np.array(df_session['frame_seconds'].values[0])
    y0 = np.array(df_session['value'].values[0])
    ax1.plot(x,y0,label = labs[0], color = colors0[0],linewidth=3)
    y1 = np.array(df_session['value'].values[1])
    ax1.plot(x,y1,label = labs[1], color = colors1[0],linewidth=3)
    #now, plot the cumulative sum
    y_cumsum0 = np.array(df_session['derivative_sign'].values[0])
    ax2.plot(x[1:],y_cumsum0,label = labs[0], color = colors0[0], linewidth=3)
    y_cumsum1 = np.array(df_session['derivative_sign'].values[1])
    ax2.plot(x[1:],y_cumsum1,label = labs[1], color = colors1[0], linewidth=3)

    
    #ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Interpersonal Rating')
    #set scale from -7 to 7
    ax1.set_ylim([-7,8])
    #set yticks
    ax1.set_yticks([-7,-3,0,3,7])
    #add horizontal dashed line at 0
    ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    #vertical lines in both, every 100 seconds
    for i in range(0,x.max().astype(int),100):
        ax1.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
        ax2.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
    #set minimum x value to 0, but no upper bound
    ax1.set_xlim([0,None])

    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Derivative IR')
    #set minimum x value to 0, but no upper bound
    ax2.set_xlim([0,None])

    ax1.legend()

    #tight layout for both axes
    plt.tight_layout()

    print(session)
    plt.show()  
    #save with high quality in png
    #fig.savefig('figures/supp' + str(session) + '.png', dpi=500)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
#plot just sessions 41 an 38, as a figure with 4 subplots
import tikzplotlib

sns.set(style="ticks", font_scale=2)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']

#create folder figures
if not os.path.exists('figures'):
    os.makedirs('figures')


sessions = df_interp['session'].unique()
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    #for each participant
    participants = df_session['participant_number'].unique()
    fig, (ax1,ax2,ax4, ax3) = plt.subplots(4,1,figsize = (12,15))  

    x = np.array(df_session['frame_seconds'].values[0])
    y0 = np.array(df_session['value'].values[0])
    ax1.plot(x,y0,label = labs[0], color = colors0[0],linewidth=3)
    y1 = np.array(df_session['value'].values[1])
    ax1.plot(x,y1,label = labs[1], color = colors1[0],linewidth=3)
    #now, plot the cumulative sum
    y_cumsum0 = np.array(df_session['cumsum'].values[0])
    ax2.plot(x,y_cumsum0,label = labs[0], color = colors0[0],alpha = 1.0, linewidth=3)
    y_cumsum1 = np.array(df_session['cumsum'].values[1])
    ax2.plot(x,y_cumsum1,label = labs[1], color = colors1[0],alpha = 1.0, linewidth=3)

    y_cumsum_sum = y_cumsum0 + y_cumsum1
    y_cumsum_sum_time = y_cumsum_sum / x #WONDERING if we should have time as a unit rather than seconds
    
    y_dif = np.abs(y0 - y1)
    y_all = y_cumsum_sum_time - y_dif
    ax3.plot(x,y_dif, color = colors2[0],linewidth=3, label = 'dif', alpha = 0.5)
    ax3.plot(x,y_all, color = colors3[0],linewidth=3, label = 'idc')

    ax4.plot(x,y_cumsum_sum_time, color = colors3[0],linewidth=3)

    
    #ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Interpersonal Rating')
    #set scale from -7 to 7
    ax1.set_ylim([-7,8])
    #set yticks
    ax1.set_yticks([-7,-3,0,3,7])
    #add horizontal dashed line at 0
    ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    #vertical lines in both, every 100 seconds
    for i in range(0,x.max().astype(int),100):
        ax1.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
        ax2.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
    #set minimum x value to 0, but no upper bound
    ax1.set_xlim([0,None])

    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Cumulative IR')
    #set minimum x value to 0, but no upper bound
    ax2.set_xlim([0,None])

    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Difference')
    ax3.legend()

    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('CIR_sum')



    ax1.legend()

    #tight layout for both axes
    plt.tight_layout()


    #tq_temp['condition'] == 'datadriven']
    #    plt.boxplot(df_postq_tempdd[col], labels = ['DataL'], positions = [ind*2+0.9],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors1[0], color=colors1[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
    plt.tight_layout()
    #sns.despine()
    print(session)
    plt.show()  
    #save with high quality in png
    #fig.savefig('figures/supp' + str(session) + '.png', dpi=500)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
#add slopes
import tikzplotlib

#open data.csv
datadf = pd.read_csv('data.csv')
#get sessions
sessionsdata = datadf['session'].unique()
print(sessionsdata)
#make column "session" into int 
datadf['session'] = datadf['session'].astype(int)

#open 4445merged
df_slopes = pd.read_csv('4445_merged_gt.csv')
df_slopes['session'] = df_slopes['session'].astype(int)

sns.set(style="ticks", font_scale=2)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"


colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['Participant A','Participant B']

#create folder figures
if not os.path.exists('figures'):
    os.makedirs('figures')


sessions = df_interp['session'].unique()
sessions = [22]
for session in sessions:
    df_session = df_interp[df_interp['session']==session]
    session2 = 2*session + 1
    sessioname = str(2*session) + str(session2)
    sessioname = int(sessioname)
    print(sessioname)
    df_data = datadf[datadf['session']==sessioname]
    print(df_data.shape)
    dfslopes = df_slopes[df_slopes['session']==sessioname]
    print(dfslopes.shape)
    x_slopes = dfslopes['timeelapsed'].values

    y_slopes0 = dfslopes['slope_p0'].values
    y_slopes1 = dfslopes['slope_p1'].values

    x_data = df_data['timeelapsed'].values
    y_data = df_data['groundtruth'].values
    
    #remove that element
    remove_el = np.where(np.diff(x_data) <= 0)[0]
    print(remove_el)
    x_data = x_data[:remove_el[0]+1]
    y_data = y_data[:remove_el[0]+1]
    print(x_data.shape)
    #get only x_slopes > than min x_data
    
    y_slopes0 = y_slopes0[x_slopes >= x_data.min()]
    y_slopes1 = y_slopes1[x_slopes >= x_data.min()]
    x_slopes = x_slopes[x_slopes >= x_data.min()]

    print(x_slopes.shape, x_data.shape)

    #for each participant
    participants = df_session['participant_number'].unique()
    fig, ((ax1,ax3), (ax2,ax4)) = plt.subplots(2,2,figsize = (25,12))  

    x = np.array(df_session['frame_seconds'].values[0])
    y0 = np.array(df_session['value'].values[0])
    ax1.plot(x,y0,label = labs[0], color = colors0[0],linewidth=3)
    y1 = np.array(df_session['value'].values[1])
    ax1.plot(x,y1,label = labs[1], color = colors1[0],linewidth=3)
    #now, plot the cumulative sum
    y_cumsum0 = np.array(df_session['cumsum'].values[0])
    ax2.plot(x,y_cumsum0,label = labs[0], color = colors0[0],alpha = 1.0, linewidth=3)
    y_cumsum1 = np.array(df_session['cumsum'].values[1])
    ax2.plot(x,y_cumsum1,label = labs[1], color = colors1[0],alpha = 1.0, linewidth=3)

    slope, intercept, r_value, p_value, std_err = stats.linregress(x,y_cumsum0)
    line = slope*x+intercept
    #ax2.plot(x,line,'--',color = colors2[0],linewidth=3,label = 'R\u00b2 : ' + str(round(r_value, 4)))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x,y_cumsum1)
    line = slope*x+intercept
    #ax2.plot(x,line,'--',color = colors3[0],linewidth=3, label = 'R\u00b2 : ' + str(round(r_value, 4)))

    #ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Interpersonal Rating')
    #set scale from -7 to 7
    ax1.set_ylim([-7,8])
    #set yticks
    ax1.set_yticks([-7,-3,0,3,7])
    #add horizontal dashed line at 0
    ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    #also for ax2 and ax3
    ax2.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    ax3.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)
    ax4.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)

    ax3.plot(x_slopes,y_slopes0, color = colors0[0],linewidth=3)
    ax3.plot(x_slopes,y_slopes1, color = colors1[0],linewidth=3)
    ax4.plot(x_data,y_data, color = colors3[0],linewidth=3)



    #vertical lines in both, every 100 seconds
    for i in range(0,x.max().astype(int),100):
        ax1.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
        ax2.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
        ax3.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
        ax4.axvline(x=i, color='k', linestyle='--',linewidth=1, alpha = 0.3)
    #set minimum x value to 0, but no upper bound
    ax1.set_xlim([0,None])

    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Cumulative IR ' + r'($y_c$)')
    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('IDC')

    #CREATE LEGEND WITH MATH y_s (Slope of CIR ($y_s$))
    ax3.set_ylabel('Slope of CIR ' +r'($y_s$)')
    #set minimum x value to 0, but no upper bound

    ax1.legend()

    #tight layout for all axes, minimum 0
    ax1.set_xlim([0,None])
    ax2.set_xlim([0,None])
    ax3.set_xlim([0,None])
    ax4.set_xlim([0,None])
    plt.tight_layout()


    #tq_temp['condition'] == 'datadriven']
    #    plt.boxplot(df_postq_tempdd[col], labels = ['DataL'], positions = [ind*2+0.9],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors1[0], color=colors1[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
    plt.tight_layout()
    #sns.despine()
    plt.show()  
    #save with high quality in png
    fig.savefig('figures/gt_' + str(session) + '.png', dpi=600)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
#now open the csvs in folder final plot

sns.set(style="ticks", font_scale=2.5)
#import itertools
plt.rcParams["font.family"] = "Times New Roman"

folder = './final_plot'
y_testdf = pd.read_csv(folder + '/y_test.csv', index_col = 0)
print(y_testdf)
#remove the first 9 rows
y_testdf = y_testdf.iloc[9:]
y_test = y_testdf['0'].values
#make first column into index
#y_testdf = y_testdf.set_index('Unnamed: 0')
y_preddf = pd.read_csv(folder + '/y_testpred.csv',index_col = 0)
y_pred = y_preddf['0'].values
#make first column into index
#y_pred = y_pred.set_index('Unnamed: 0')
print(y_test.shape, y_pred.shape)

#now, plot both

colors0 = ['royalblue']#,'mediumblue','cornflowerblue']
colors1 = ['indianred']#,'firebrick']
colors2 = ['darkgreen']#,'forestgreen','limegreen']
colors3 = ['darkorange']#,'goldenrod','gold']

labs = ['ICD','Predicted IDC']

#create folder figures
if not os.path.exists('figures'):
    os.makedirs('figures')

fig,ax = plt.subplots(figsize = (25,12))  

x = np.array(y_preddf.index.values[750:1000])
x = x - x[0]
ax.plot(x,y_test[750:1000],label = labs[0], color = colors3[0],linewidth=5)
ax.plot(x,y_pred[750:1000],label = labs[1], color = colors2[0],linewidth=5)
ax.set_ylabel('IDC')
#set scale from -7 to 7
#ax1.set_ylim([-7,8])
#set yticks
#ax1.set_yticks([-7,-3,0,3,7])
#add horizontal dashed line at 0
#ax1.axhline(y=0, color='k', linestyle='--',linewidth=1, alpha = 0.5)

#ax1.set_xlim([0,None])

ax.set_xlabel('Samples')
ax.legend()

#tight layout for all axes, minimum 0
#ax.set_xlim([0,None])
plt.tight_layout()


plt.show()  
fig.savefig('figures/results.png', dpi=600)

#tikzplotlib.save("results2.tex")

#for ind,col in enumerate(cols_oi):
#    df_postq_tempn = df_postq_temp.loc[df_postq_temp['condition'] == 'naive']
#    plt.boxplot(df_postq_tempn[col], labels = ['NaïveL'], positions = [ind*2 + 0.1],  widths = 0.6, patch_artist=True, boxprops=dict(facecolor=colors0[0], color=colors0[0]), medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'))
#    df_postq_tempdd = df_postq_temp.loc[df_pos

In [ ]:
r_values =np.array([0.95059199, 0.99983174, 0.9036017, 0.9144214, 0.99462086, 0.97762785, 0.99980018, 0.99982248, 
                    0.90478528, -0.6934582, 0.99351582, 0.99894149, 0.99398677, 0.79835396, 0.97484511, 0.99706833,
                      0.47062635, 0.99774366, 0.99213275, 0.99629849, 0.89715595, 0.99479227, 0.99199593, 0.99646349])
print(np.mean(r_values))
print(np.std(r_values))

In [ ]:
#now, plot some of the datapoints for each session, from df_disagreement
cols_oi = ['mse', 'corr', 'abs_diff', 'cohen_kappa', 'valuesum',
       'interpersonal_diff', 'agreeable_general_sum', 'likingscale_sum',
       'extraversion_diff', 'agreeableness_diff', 'conscientiousness_diff',
       'neuroticism_diff', 'openness_diff', 'religiousness_diff',
       'political_diff', 'agreeable_general_diff', 
       'likingscale_diff', 'slope_diff']

#MISSING THE RELATIONALC_DIFF

#sessions
sessions = df_disagreement['session'].unique()
#get a color palette with the number of sessions we have
colors = sns.color_palette("hls", len(cols_oi))
fig,ax = plt.subplots(figsize=(15,5))
#for ind,session in enumerate(sessions):
#    #plot each column in the same column
#    df_session = df_disagreement[df_disagreement['session']==session]
#    for indx,col in enumerate(cols_oi):
#        sns.scatterplot(x=ind, y = df_session[col],ax=ax,label = col,color=colors[indx])

for indx,col in enumerate(cols_oi):
        sns.lineplot(x=np.arange(len(sessions)), y = df_disagreement[col].values,ax=ax,label = col,color=colors[indx])
    

In [ ]:
#now, use pearson correlation to find which variables better correlate with the slope difference
df_disagreement_numeric = df_disagreement[cols_oi].astype(float)
df_corr = df_disagreement_numeric.corr(method='pearson')
pval = df_disagreement_numeric.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*df_corr.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [.05, .01, .001] if x<=t]))
df_corr_pval = df_corr.round(2).astype(str) + p
df_corr_pval

In [ ]:
#plot df_corr as a correlation matrix
fig, ax = plt.subplots(figsize=(15,10))
sns.heatmap(df_corr, annot=True, ax=ax)


In [ ]:
#print participant demographics
#avg age + std
df_midq['age'] = df_midq['age'].astype(float)
print('Average age: ',df_midq['age'].mean(), ' +- ', df_midq['age'].std())
#age range
print('Age range: ',df_midq['age'].min(), ' - ', df_midq['age'].max())
#gender distribution - count occurrences of every value
print( df_midq['gender'].value_counts())
#race distribution
print(df_midq['race'].value_counts())
print(df_midq['proficiency'].value_counts())
df_midq

In [ ]:
cols_midq = ['extraversion', 'agreeableness', 'conscientiousness',
       'neuroticism', 'openness', 'religiousness', 'political',
       'agreeable_general', 'relationalc', 'likingscale', 'interpersonal','slope']

#get correlation matrix
df_midq_numeric = df_midq[cols_midq].astype(float)
df_corr = df_midq_numeric.corr(method='pearson')
pval = df_midq_numeric.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(*df_corr.shape)
p = pval.applymap(lambda x: ''.join(['*' for t in [.05, .01, .001] if x<=t]))
df_corr_pval = df_corr.round(2).astype(str) + p
df_corr_pval

In [ ]:
#plot
fig, ax = plt.subplots(figsize=(10,10))
sns.heatmap(df_corr, annot=True, ax=ax)


# LABEL DISTRIBUTION


In [ ]:
#imports
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.interpolate import interp1d
#import mse
from sklearn.metrics import mean_squared_error
import scipy.stats as stats
import itertools
from scipy.stats import pearsonr
from scipy.integrate import trapz
#import tikzplotlib#

#for plots
import matplotlib as mpl

sns.set_theme(style="whitegrid", font_scale=2)#, font='Arial')
#sns.set_palette("husl")
#plt.rcParams.update({'font.size': '50'})
#plt.rcParams['font.size'] = '50'
#mpl.rcParams['font.family'] = 'Arial'

from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
#from termcolor import colored

#open df_interp
df_interp = pd.read_pickle('df_interpolated5dec.pkl')


#trying this with 0.5s of interpolation, which seems like enough for a system monitoring IDC


In [ ]:
df_interp

In [ ]:
#go row by row, save values of list inder "values" in a single list with all the values for all the participants

#for row in row
all_values = []
for ind,row in df_interp.iterrows():
    #print(row['value'])
    #get the list of values
    values_list = row['value']
    #print(values_list)
    all_values.extend(values_list)

#now, plot the histogram of all values
fig, ax = plt.subplots(figsize=(20,10))
sns.histplot(all_values, kde=True, ax=ax)
ax.set_xlabel('Ratings')



plt.show()  
fig.savefig('label_dist_discrete.png', dpi=600)
#sns.set_palette("husl")
    

In [ ]:
#now, get the derivative of these values and print



#for row in row
all_values = []
for ind,row in df_interp.iterrows():
    #print(row['value'])
    #get the list of values
    values_list = row['value']
    #print(values_list)
    all_values.extend(values_list)

    #plot the histogram of all values  
print(len(all_values))
print('length of all derivatives', len(all_values))
print('positive vs negative values')
print(len([x for x in all_values if x > 0]), len([x for x in all_values if x < 0]))
print('positive to negative ratio')
print(len([x for x in all_values if x > 0])/len([x for x in all_values if x < 0]))
fig, ax = plt.subplots(figsize=(10,10))
sns.histplot(all_values, kde=False, ax=ax)
ax.set_xlabel('Ratings')



plt.show()  
fig.savefig('label_dist_discrete.png', dpi=600)
#sns.set_palette("husl")

In [ ]:
#now, get the derivative of these values and print


all_derivatives = []
for ind,row in df_interp.iterrows():
    #print(row['value'])
    #get the list of values
    values_list = row['derivative']
    print(values_list)
    all_derivatives.extend(values_list)

    #plot the histogram of all values  
print(len(all_derivatives))
fig, ax = plt.subplots(figsize=(20,10))
sns.histplot(all_derivatives, kde=False, ax=ax)
ax.set_xlabel('Ratings')



plt.show()  
fig.savefig('label_dist_sign_0.png', dpi=600)
#sns.set_palette("husl")   

In [ ]:
#now, get the derivative of these values and print


all_derivatives = []
for ind,row in df_interp.iterrows():
    #print(row['value'])
    #get the list of values
    values_list = row['derivative']
    #print(values_list)
    all_derivatives.extend(values_list)

    #plot the histogram of all values  
print(len(all_derivatives))
#remove the 0
all_derivatives = [x for x in all_derivatives if x != 0]
print('length of all derivatives', len(all_derivatives))
print('positive vs negative values')
print(len([x for x in all_derivatives if x > 0]), len([x for x in all_derivatives if x < 0]))
print('positive to negative ratio')
print(len([x for x in all_derivatives if x > 0])/len([x for x in all_derivatives if x < 0]))
fig, ax = plt.subplots(figsize=(20,10))
sns.histplot(all_derivatives, kde=False, ax=ax)
ax.set_xlabel('Ratings')
ax.set_xticklabels(np.arange(-10,10,2))


plt.show()  
fig.savefig('label_dist_sign.png', dpi=600)
#sns.set_palette("husl")   